# 2장 04. 준비된 metric 기록 읽기

이 notebook은 직접 계산한 metric과 별도로, 준비된 평가 기록 파일에 어떤 값이 남는지 확인합니다.


## 기본 개념: metric record와 experiment tracking

Metric 기록은 단순히 성능 숫자를 저장하는 파일이 아닙니다. MLOps에서 metric record는 어떤 데이터셋, 어떤 feature 목록, 어떤 label 기준, 어떤 model version, 어떤 threshold에서 나온 결과인지 남기는 재현성 증거입니다.

MLflow 같은 experiment tracking 도구는 이 정보를 run 단위로 묶습니다. 실무에서는 여러 모델 후보가 생기기 때문에 사람이 파일명만 보고 어떤 모델이 더 나은지 판단하면 위험합니다. run, param, metric, artifact가 함께 남아야 비교가 가능합니다.

이 과정에서는 MLflow server가 없어도 JSON 평가 기록을 필수 근거로 사용합니다. MLflow는 선택 실행 경로일 수 있지만, 기록해야 하는 정보는 선택 사항이 아닙니다.

| 기록 항목 | 왜 필요한가 | 예시 |
| --- | --- | --- |
| dataset | 어떤 데이터로 평가했는지 구분 | `vital_signs_test` |
| model version | 어떤 후보인지 구분 | `v1` |
| threshold | prediction 변환 기준 추적 | `0.5` |
| metric | 후보 비교 기준 | Precision, Recall, FP, FN |
| artifact | 실제 모델 파일 또는 package 위치 | model path, MLflow artifact URI |

이 노트북에서 읽은 기록은 3장의 MLflow model URI와 연결됩니다. 즉 “좋아 보이는 모델 파일”이 아니라 “평가 조건이 남아 있는 배포 후보”를 serving 단계로 넘기는 구조를 봅니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
import json
from pathlib import Path

path = Path("artifacts/experiments/chapter_02/model_test_eval.json")
if not path.exists():
    path = Path("../artifacts/experiments/chapter_02/model_test_eval.json")
if not path.exists():
    path = Path("../../artifacts/experiments/chapter_02/model_test_eval.json")

record = json.loads(path.read_text())
print("path:", path)


## 1. 기록에 있는 주요 값 보기

기록 파일에서는 모델 버전, threshold, metric 값을 확인합니다.


In [ ]:
record.keys()


In [ ]:
rows = []
for key in ["accuracy", "precision", "recall", "f1"]:
    if key in record:
        rows.append({"metric": key, "value": record[key]})

pd.DataFrame(rows)
